# Retinal Disease Classification - Full Training (Google Colab)

Complete end-to-end training notebook for diabetic retinopathy classification.

**Models:** ViT-B/16, ResNet-50, EfficientNet-B4  
**Dataset:** APTOS 2019 Blindness Detection (Kaggle)

## Step 1: Setup GPU & Dependencies

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("❌ GPU not available. Go to Runtime → Change runtime type → GPU (T4)")

# Clone repository
REPO_URL = "https://github.com/umerkhan-12/dlp_project.git"
REPO_DIR = Path("/content/dlp_project")

if not REPO_DIR.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Ensure kaggle is installed
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)

print("✓ Dependencies installed")

## Step 2: Configure Kaggle Credentials

In [ ]:
import json

# Set your Kaggle credentials below
KAGGLE_USERNAME = "k230798umerkanthi"
KAGGLE_KEY = "KGAT_3df93fc7685a9da792cd51467f399341"

# Save to kaggle.json
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)

kaggle_config_path = kaggle_dir / "kaggle.json"
kaggle_config = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}

with open(kaggle_config_path, "w", encoding="utf-8") as f:
    json.dump(kaggle_config, f)

kaggle_config_path.chmod(0o600)

print(f"✓ Kaggle config saved to {kaggle_config_path}")
print(f"  Username: {KAGGLE_USERNAME}")
print(f"  Key: {KAGGLE_KEY[:10]}...")

## Step 3: Verify Kaggle Setup

In [ ]:
# Test Kaggle
print("Testing Kaggle credentials...")

# Check if kaggle can authenticate
result = subprocess.run(
    ["kaggle", "competitions", "list"],
    capture_output=True,
    text=True,
    timeout=30,
)

if result.returncode == 0:
    print("✓ Kaggle authentication successful")
    competitions = result.stdout.strip().split("\n")[:3]
    print("  Sample competitions:")
    for comp in competitions:
        print(f"    - {comp}")
else:
    print("❌ Kaggle authentication failed!")
    print("Error:", result.stderr)
    print("\n⚠️  CRITICAL STEPS:")
    print("1. Go to https://www.kaggle.com/competitions/aptos2019-blindness-detection")
    print("2. Click 'Accept Rules' (required!)")
    print("3. Check your credentials are correct")
    print("4. Try again")
    raise RuntimeError("Kaggle authentication failed")

## Step 4: Download APTOS Dataset

In [ ]:
import pandas as pd
import zipfile

data_dir = REPO_DIR / "data" / "raw"
aptos_dir = data_dir / "aptos2019-blindness-detection"
train_csv = aptos_dir / "train.csv"
images_dir = aptos_dir / "train_images"

# Skip if already exists
if train_csv.exists() and images_dir.exists():
    print("✓ Dataset already exists, skipping download")
else:
    print("Downloading APTOS 2019 dataset...")
    data_dir.mkdir(parents=True, exist_ok=True)
    
    # Download
    result = subprocess.run(
        ["kaggle", "competitions", "download", "-c", "aptos2019-blindness-detection", "-p", str(data_dir)],
        capture_output=True,
        text=True,
        timeout=600,
    )
    
    if result.returncode != 0:
        print("❌ Download failed:")
        print(result.stderr)
        raise RuntimeError("Dataset download failed. Check Kaggle credentials and competition acceptance.")
    
    print(result.stdout)
    
    # Extract
    zip_path = data_dir / "aptos2019-blindness-detection.zip"
    if zip_path.exists():
        print(f"Extracting {zip_path.name}...")
        aptos_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(aptos_dir)
        zip_path.unlink()
        print(f"✓ Extracted to {aptos_dir}")

# Verify
if not train_csv.exists():
    print(f"❌ train.csv not found at {train_csv}")
    print(f"   aptos_dir contents: {list(aptos_dir.glob('*')) if aptos_dir.exists() else 'dir not found'}")
    raise FileNotFoundError("Dataset extraction failed")

if not images_dir.exists():
    print(f"❌ train_images dir not found at {images_dir}")
    raise FileNotFoundError("Images directory not found")

# Summary
df = pd.read_csv(train_csv)
image_count = len(list(images_dir.glob("*.png")))

print(f"\n✓ Dataset verified:")
print(f"  Samples: {len(df)}")
print(f"  Images: {image_count}")
print(f"  Classes: {df['diagnosis'].nunique()}")

## Step 5: Setup Training Output

In [ ]:
# Optional: Use Google Drive for persistent storage
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)
    OUTPUT_DIR = Path('/content/drive/MyDrive/retinal_disease_full_training')
    print(f"✓ Google Drive mounted")
else:
    OUTPUT_DIR = REPO_DIR / 'experiments'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Training configurations
TRAIN_PLAN = [
    {"model": "vit_base_patch16_224", "epochs": 100, "batch_size": 16, "lr": "3e-4"},
    {"model": "resnet50", "epochs": 60, "batch_size": 32, "lr": "3e-4"},
    {"model": "efficientnet_b4", "epochs": 60, "batch_size": 20, "lr": "2e-4"},
]

print("\nTraining plan:")
for item in TRAIN_PLAN:
    print(f"  - {item['model']:25} {item['epochs']:3} epochs, batch={item['batch_size']}, lr={item['lr']}")

## Step 6: Train All Models

In [ ]:
DATA_DIR = REPO_DIR / "data" / "raw" / "aptos2019-blindness-detection"

for i, item in enumerate(TRAIN_PLAN, 1):
    print(f"\n{'='*80}")
    print(f"TRAINING MODEL {i}/{len(TRAIN_PLAN)}: {item['model']}")
    print(f"{'='*80}")
    
    cmd = [
        sys.executable, "main.py",
        "--data-dir", str(DATA_DIR),
        "--output-dir", str(OUTPUT_DIR),
        "--model", item["model"],
        "--epochs", str(item["epochs"]),
        "--batch-size", str(item["batch_size"]),
        "--lr", item["lr"],
        "--device", "cuda",
        "--num-workers", "4",
        "--early-stopping", "15",
        "--mixed-precision",
        "--experiment-name", "colab_full",
    ]
    
    result = subprocess.run(cmd, cwd=REPO_DIR)
    
    if result.returncode != 0:
        print(f"\n⚠️  Training for {item['model']} failed or interrupted")
        print("   Continuing with next model...")
    else:
        print(f"\n✓ {item['model']} training completed")

print(f"\n{'='*80}")
print("ALL TRAININGS COMPLETED")
print(f"{'='*80}")

## Step 7: Summarize Results

In [ ]:
import json
import re
import pandas as pd
import torch

def parse_metrics_file(path: Path):
    """Parse metrics from text file."""
    metrics = {}
    if not path.exists():
        return metrics
    for line in path.read_text().splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip().lower().replace(" ", "_").replace("(", "").replace(")", "")
        value = value.strip()
        if re.fullmatch(r"[-+]?\d*\.?\d+", value):
            metrics[key] = float(value)
    return metrics

# Collect results
rows = []
for exp_dir in sorted([p for p in OUTPUT_DIR.glob("*") if p.is_dir()]):
    config_path = exp_dir / "config.json"
    ckpt_path = exp_dir / "checkpoints" / "best_model.pth"
    metrics_path = exp_dir / "results" / "test_metrics.txt"
    
    if not config_path.exists():
        continue

    cfg = json.loads(config_path.read_text())
    best_epoch = None
    
    if ckpt_path.exists():
        try:
            ckpt = torch.load(ckpt_path, map_location="cpu")
            best_epoch = int(ckpt.get("epoch", -1)) + 1
        except:
            best_epoch = "error"

    metrics = parse_metrics_file(metrics_path)
    
    rows.append({
        "Model": cfg.get("model", "?"),
        "Epochs": cfg.get("epochs"),
        "Best Epoch": best_epoch,
        "Accuracy": f"{metrics.get('accuracy', 0):.4f}" if "accuracy" in metrics else "N/A",
        "Balanced Acc": f"{metrics.get('balanced_accuracy', 0):.4f}" if "balanced_accuracy" in metrics else "N/A",
        "F1 (Macro)": f"{metrics.get('f1_macro', 0):.4f}" if "f1_macro" in metrics else "N/A",
        "Kappa": f"{metrics.get('quadratic_kappa', 0):.4f}" if "quadratic_kappa" in metrics else "N/A",
    })

if rows:
    summary_df = pd.DataFrame(rows)
    print("\n📊 TRAINING SUMMARY")
    print(summary_df.to_string(index=False))
else:
    print("No completed trainings found")

print(f"\nResults saved to: {OUTPUT_DIR}")

## Step 8: (Optional) Download Results

In [ ]:
# Create archive of all results
import shutil

archive_base = Path('/content/retinal_disease_results')
archive_path = shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=str(OUTPUT_DIR),
)

archive_size_mb = Path(archive_path).stat().st_size / (1024**2)
print(f"\n✓ Results archived: {archive_path}")
print(f"  Size: {archive_size_mb:.1f} MB")

if archive_size_mb < 2000:  # Under 2GB, safe to download
    print("\nTo download: Run the cell below")
else:
    print(f"\n⚠️  Archive is {archive_size_mb:.0f} MB (too large for direct download)")
    print("    Use Google Drive instead (already saved)")

In [ ]:
# Optional: Download results to your computer
# Uncomment to download

# from google.colab import files
# files.download('/content/retinal_disease_results.zip')